# Sentimento de textos sobre criptoativos

Três classes: negativo, neutro, positivo. O BERT base (inglês, sem caixa)
é ajustado nos textos em português que o detector de idioma deixa passar.
O vocabulário inglês corta palavras portuguesas em pedaços; o ajuste fino
compensa parte disso nos pesos da classificação.

Desempenho: o `Trainer` usa a GPU se o PyTorch enxergar CUDA, com precisão
mista (`fp16`) na RTX 3050. O padding deixa de ir até 512 tokens em todo
exemplo e passa a completar só até o maior texto do lote. Isso é o que
deixa o ajuste mais curto. Sem GPU o mesmo código treina na CPU, sem `fp16`.

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from langid import classify
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

cuda = torch.cuda.is_available()
print("cuda:", cuda, torch.cuda.get_device_name(0) if cuda else "")

## Limpeza

Ficam só textos classificados como português. Saem URL, hashtag, menção e dígitos.
O mapa de rótulos é fixo: Negativo 0, Neutro 1, Positivo 2.

In [ ]:
df = pd.read_csv("dataset.csv")
df["lang"] = [classify(t)[0] if isinstance(t, str) else "desconhecido" for t in df["texto"]]
df = df.loc[df["lang"] == "pt"].copy()
df["texto"] = (
    df["texto"]
    .str.replace(r"http\S+|#\w+|@\w+|\d+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
df["label"] = df["sentimento"].map({"Negativo": 0, "Neutro": 1, "Positivo": 2})
print(df["sentimento"].value_counts())
print("linhas em português:", len(df))

## Ajuste

`padding` não vai no tokenizador. O `DataCollatorWithPadding` iguala o lote
na hora do treino. A divisão é 80/20. Dez épocas, taxa 2e-5, lote 8 e
acúmulo de 2 passos (equivale a um lote de 16 sem estourar os 6 GB).

In [ ]:
def metricas(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, pred, average="weighted", zero_division=0)
    return {
        "accuracy": accuracy_score(labels, pred),
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
modelo = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)

dataset = Dataset.from_pandas(df[["texto", "label"]])
tokenizado = dataset.map(
    lambda lote: tokenizer(lote["texto"], truncation=True, max_length=128),
    batched=True,
)
partes = tokenizado.train_test_split(test_size=0.2, seed=64)

treino = Trainer(
    model=modelo,
    args=TrainingArguments(
        output_dir="./resultados",
        eval_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=10,
        weight_decay=0.01,
        gradient_accumulation_steps=2,
        logging_steps=10,
        fp16=cuda,
        save_strategy="no",
        report_to="none",
    ),
    train_dataset=partes["train"],
    eval_dataset=partes["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=metricas,
)
treino.train()
print(treino.evaluate())
modelo.save_pretrained("./dsa_modelo_final")
tokenizer.save_pretrained("./dsa_modelo_final")